# Week 2: Similarity, Clustering, and Stable Patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/02/Week_02_Similarity_Clustering_Stable_Patterns.ipynb)

**Advanced Data Mining · Master level · provisional 180-minute core session**

## Same customers, different stories

Last week, cleaning choices changed our descriptive findings. Today, we keep the eligible purchases fixed and ask who appears similar to whom. **Can we defend a useful description of these customers, or are we mostly seeing the choices we made?** We will follow a few real customer profiles from nearest neighbors to groups, then challenge the grouping.

From the previous course, remember that a representation preserves some differences and suppresses others. The same principle applies to three customer features as to a learned embedding. A convincing picture begins an investigation; the original records and a controlled comparison help us explain it.

By the end, you should be able to:

1. Construct customer features with explicit observation and purchase policies.
2. Explain and test how units, transformations, and distance change similarity.
3. Interpret customer groups through original-unit profiles and inspected cases.
4. Distinguish separation, stability, and usefulness, and qualify a segmentation claim.

| Block | Working time, excluding breaks |
|---|---:|
| Opening prediction and setup | 10 min |
| Reading A and tiny experiments | 20 min |
| Independent investigation 1, including customer features | 40 min |
| Reading B and algorithm demonstrations | 25 min |
| Reading C and a worked comparison | 20 min |
| Independent investigation 2 | 40 min |
| Decision memos, discussion, and synthesis | 25 min |
| Total | 180 min |

Readings A–C each offer 10–15 minutes of independent reading and reflection within their block. Choose one route per investigation. The notebook runs independently of Week 1 and rebuilds the same positive-sales policy. The activities are formative practice.

### Before calculating: which pair seems similar?

These are invented profiles for a thought experiment, not records from the retailer.

| Profile | Days since last purchase | Positive invoices | Positive purchase value |
|---|---:|---:|---:|
| A | 5 | 12 | £600 |
| B | 7 | 11 | £6,000 |
| C | 180 | 2 | £620 |

Choose a pair for each question: who has a similar purchase rhythm, and who has a similar total purchase value? Write one sentence explaining each choice. You can disagree with another reader because these questions preserve different differences. We will make that choice explicit in the code.

## Environment and getting started

**Colab:** click **Open in Colab** above, connect to a CPU runtime, and choose **Runtime → Run all**. The setup cell installs the required packages; the data cell downloads and verifies the small course dataset automatically. No upload, ZIP extraction, account token, or manual data preparation is required. If Colab asks for a runtime restart after installation, restart and choose Run all again.

**Local:** from the course root, run `uv sync --locked`, then `uv run jupyter lab`. Open this notebook with the environment's Python kernel and run cells in order. The supplied `data/` file is used locally; otherwise the same download runs automatically. First-time installation/download needs internet. Afterward, local cached data is reused.

**Prerequisites:** Python variables, pandas tables, basic plots, and the previous course's idea that a representation determines what information is available. CPU is sufficient; no neural-network training is required today.

Run the baseline first. During the investigations, change one setting and rerun its relevant cells. Restart and run all if changes leave you unsure which variables are current.

In [ ]:
# Install automatically in Colab; local environments are managed with uv.
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'numpy>=2,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8'
    ])
    print('Colab dependencies ready. Continue with the next cell.')
else:
    print('Local runtime: using the course environment (uv sync --locked).')

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
pd.set_option('display.max_columns', 12)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
print('NumPy:', np.__version__, '| pandas:', pd.__version__)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from sklearn.datasets import make_moons
import sklearn
print('scikit-learn:', sklearn.__version__)

## Reading A — What does “similar” mean?

**Read and reflect independently for 10–15 minutes.** Start with the opening profiles and your two choices. By the end, you should be able to explain a distance in terms of the differences it notices.

### A representation answers part of the question

Suppose a colleague asks you to find customers like customer A. Before calculating, ask what the comparison will support. A customer who visits regularly may resemble A in purchasing rhythm while spending ten times more. Another customer may spend almost the same total in two large orders, then disappear for months. Both are defensible meanings of “like,” but they suggest different questions to investigate.

A representation turns each observation into a list of measurements. In this notebook, one observation will be a customer ID and its measurements will be recency, frequency, and monetary value. A distance then combines differences between those measurements into one number. It cannot consider information absent from the representation. Two customers with the same three totals might buy completely different products, or one might return most purchases. Their RFM vectors would still match under our positive-sales policy.

This is the connection to representation learning: the method can use the distinctions that the vector preserves. Here we can inspect every feature directly. That makes the consequences of our choices easier to see before we later work with richer text representations.

### A distance needs a ruler

For vectors $x$ and $y$ with $p$ features, Euclidean distance is

$$d(x,y)=\sqrt{\sum_{j=1}^{p}(x_j-y_j)^2}.$$

Each $j$ identifies one feature. Identical vectors have distance zero. Squaring prevents positive and negative differences from canceling and gives larger differences more influence. The square root converts the sum of squares back to a distance. Combining days, invoice counts, and pounds without a scaling decision produces a numerical distance whose interpretation depends on all those units.

Consider a query with 5 orders and £500 spend. Candidate A has 6 orders and £100 spend; candidate B has 10 orders and £480 spend. Their raw distances are approximately 400 and 20.6. Money dominates that comparison: a five-order difference barely matters beside a £400 difference. If we convert pounds to pence, we multiply the monetary difference by 100 and its squared contribution by 10,000. No customer has changed, but the numerical balance of features has.

Standardization changes the ruler for each feature:

$$z_j=(x_j-\mu_j)/\sigma_j.$$

Here $\mu_j$ and $\sigma_j$ are the feature's mean and standard deviation in a specified reference cohort. A standardized difference of one represents one cohort standard deviation. For a constant feature, there is no observed variation to distinguish customers; scikit-learn safely leaves its centered values at zero rather than dividing by zero. A future-customer evaluation should fit these statistics on the reference data and reuse them unchanged.

Standardization removes dependence on positive unit conversions when the statistics are refitted consistently. It does not establish that all features deserve equal influence. The cohort determines the ruler, and extreme observations can affect its mean and standard deviation. In the first example below, the closest customer happens to stay the same. In a second deliberately constructed example, the ranking reverses. Both outcomes are consistent with the mechanism.

### Size and pattern can tell different stories

For nonzero vectors, cosine similarity is $x^\top y/(\|x\|\|y\|)$, and cosine distance is one minus that value. The dot product combines matching coordinates; the norms remove overall length. Vectors $[1,2,0]$ and $[10,20,0]$ point in the same direction, so cosine distance is zero even though Euclidean distance is large. On nonnegative purchase vectors, cosine distance ranges from zero for the same direction to one for no overlapping coordinates. General signed vectors can have distances up to two. A zero vector has no direction, so it needs an explicit handling policy.

Cosine can help compare a product mix across different volumes. It cannot preserve absolute spending at the same time. Also, centering a vector changes its direction: cosine on centered RFM is not automatically a measure of product preference. We use a category example to establish the mechanism and Euclidean distance for today's RFM investigation.

### Check your understanding before running the examples

Write two short answers: why can a unit conversion affect raw distance, and which information would cosine lose when comparing purchase-category vectors?

<details><summary>Self-check — open after writing</summary>

A unit conversion changes a feature's numerical contribution relative to the others. Standardization can cancel the conversion, but raw distance cannot. Cosine ignores a positive common multiplier, so customers with the same category proportions can match despite very different volumes. Neither rule is universally correct; explain what the question needs.

</details>

In [ ]:
toy_customers = pd.DataFrame({'orders': [5, 6, 10, 2],
                              'spend_GBP': [500, 100, 480, 200]},
                             index=['query', 'A', 'B', 'C'])
toy_scaled = StandardScaler().fit_transform(toy_customers)
distances = pd.DataFrame({
    'raw_distance': euclidean_distances(toy_customers.iloc[[0]], toy_customers)[0],
    'standardized_distance': euclidean_distances(toy_scaled[[0]], toy_scaled)[0]}, index=toy_customers.index)
display(toy_customers); display(distances.drop('query').round(3))

# TODO 1: Predict the effect of pounds → pence, then change this setting to 100.
money_unit_multiplier = 1
converted = toy_customers.copy()
converted['spend_GBP'] *= money_unit_multiplier
converted_scaled = StandardScaler().fit_transform(converted)
display(pd.DataFrame({
    'raw_after_conversion': euclidean_distances(converted.iloc[[0]], converted)[0],
    'standardized_after_conversion': euclidean_distances(converted_scaled[[0]], converted_scaled)[0]
}, index=converted.index).drop('query').round(3))

In [ ]:
# Deliberately constructed synthetic example: watch the nearest neighbor reverse.
ranking_example = pd.DataFrame({'orders': [5, 6, 10, 2],
                                'spend_GBP': [500, 100, 480, 10000]},
                               index=['query', 'A', 'B', 'C'])
ranking_scaled = StandardScaler().fit_transform(ranking_example)
ranking_distances = pd.DataFrame({
    'raw': euclidean_distances(ranking_example.iloc[[0]], ranking_example)[0],
    'standardized': euclidean_distances(ranking_scaled[[0]], ranking_scaled)[0]
}, index=ranking_example.index).drop('query')
display(ranking_example); display(ranking_distances.round(3))
print('Nearest under each ruler:', ranking_distances.idxmin().to_dict())
# C makes the monetary standard deviation much larger. Inspect A versus B.
# Try C's spend at 200 again. Which ruler changed, and why?

### Direction versus magnitude

Cosine similarity is $\frac{x^\top y}{\|x\|\|y\|}$; cosine distance is one minus this value. For nonzero purchase-category vectors, it compares the pattern of spending while ignoring a positive common scale factor. It can therefore consider a small and a large customer identical in direction.

This is useful when category preference is the question, but loses total volume. Applying cosine distance to arbitrarily centered features can change its meaning. Today we compare it on a simple nonnegative example; customer clustering later uses Euclidean distance on transformed features.

In [ ]:
profiles = pd.DataFrame([[1, 2, 0], [10, 20, 0], [0, 2, 1]],
                        index=['small', 'same_mix_large', 'different_mix'],
                        columns=['books', 'food', 'tools'])
display(pd.DataFrame({'euclidean_from_small': euclidean_distances(profiles.iloc[[0]], profiles)[0],
                      'cosine_from_small': cosine_distances(profiles.iloc[[0]], profiles)[0]},
                     index=profiles.index).round(3))
# TODO 2: Change the large customer's category mix. Explain what each distance notices.

### Our real dataset: a reproducible retail extract

We use a supplied extract of [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), a UK online retailer's historical invoice lines. A row is a **product line**, not a customer and not necessarily an entire purchase. Prices are in pounds sterling; invoice IDs beginning with `C` mark cancellations.

The extract preserves every source row for 400 sampled customer IDs, plus rows from ten sampled anonymous invoices per calendar month. It retains source defects and cancellations. Known and anonymous records have different sampling mechanisms: their relative proportions cannot estimate missing-ID prevalence in the original population. This is a historical case study, not a claim about present-day shoppers. December 2011 is incomplete.

**Attribution:** Chen, D. (2015), *Online Retail*, UCI, [DOI: 10.24432/C5BW33](https://doi.org/10.24432/C5BW33), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course adaptation is customer/invoice selection plus CSV conversion; see `data/README.md` and `retail_manifest.json` for provenance. No synthetic defects were added to the real extract.

In [ ]:
# Prefer the bundled/local cache; otherwise fetch the public course dataset.
import urllib.request
import urllib.error

filename = 'online_retail_teaching.csv.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/online_retail_teaching.csv.gz'
expected_sha256 = '8503a734cd203ed63df3292c2a12d0b803206dc18ed67a7dc83664fd307d96d8'
candidates = [Path.cwd() / filename]
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(folder / 'data' / filename)
data_path = next((p for p in candidates if p.is_file()), None)

if data_path is None:
    data_path = Path.cwd() / 'data' / filename
    print('Downloading the course dataset (approximately 610 KB)...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            payload = response.read()
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(
            'Could not download the course dataset. Check your internet connection and retry. '
            'If the URL is unavailable, ask the instructor to check the public course release. '
            f'URL: {DATA_URL}'
        ) from error
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError('Downloaded data does not match this notebook version. Reopen the current course notebook.')
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(payload)
else:
    print('Using the cached course dataset.')

if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Cached dataset differs from this notebook version. Use a fresh Colab runtime or restore the matching course data locally.')
raw = pd.read_csv(data_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string',
                                  'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
assert len(raw) > 0 and raw.InvoiceDate.notna().all()
print(f'{len(raw):,} rows; {raw.CustomerID.nunique()} observed customer IDs')
display(raw.head(4))

## 2. A customer representation: recency, frequency, monetary value

We now change the observation unit from invoice lines to customers. **Recency** is days since the last positive purchase; **frequency** counts distinct positive invoices; **monetary value** sums positive recorded line values. These describe observed behavior within the dataset window, not lifetime value or profitability.

We exclude anonymous rows because they cannot be assigned to customers. We drop exact repeats as an explicit policy, exclude cancellations and nonpositive quantity/price, and fix the observation endpoint at **10 December 2011**, the day after the source window ends. Using today's date would make this historical exercise drift over time.

Frequency counts invoices, not product lines. Returns are not subtracted from monetary value under this policy. A high-spend customer may also have substantial returns; that is an extension, not a hidden interpretation of our feature.

In [ ]:
sales = raw.drop_duplicates().copy()
sales = sales[~sales.InvoiceNo.str.upper().str.startswith('C', na=False)]
sales = sales[(sales.Quantity > 0) & (sales.UnitPrice > 0) & sales.CustomerID.notna()].copy()
sales['LineValue'] = sales.Quantity * sales.UnitPrice
snapshot = pd.Timestamp('2011-12-10')
rfm = sales.groupby('CustomerID').agg(
    last_purchase=('InvoiceDate', 'max'), frequency=('InvoiceNo', 'nunique'),
    monetary=('LineValue', 'sum'))
rfm['recency'] = (snapshot - rfm.last_purchase).dt.days
rfm = rfm[['recency', 'frequency', 'monetary']].sort_index()
assert (rfm >= 0).all().all() and rfm.index.is_unique
display(rfm.describe().round(2))
print('Customers with positive identified sales:', len(rfm))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, column in zip(axes, rfm.columns):
    ax.hist(rfm[column], bins=25, color='#277da8')
    ax.set(xlabel=column, ylabel='Customer count')
plt.tight_layout(); plt.show()
# TODO 3: Inspect the top five monetary values. Describe the long tail without deleting it.
display(rfm.nlargest(5, 'monetary'))

## Independent investigation 1 — Find this customer's neighbors (40 minutes)

This block includes the feature construction just above and the comparisons below. Allow about 10 minutes to inspect RFM and the case customers, 20 minutes to experiment, and 10 minutes to write and compare explanations.

We compare raw features, standardized features, and `log1p` followed by standardization. `log1p(x)` means $\log(1+x)$, so zero remains defined. For large positive values it emphasizes relative differences: £10 to £100 becomes more comparable to £100 to £1,000. It still preserves ordering within a feature. This changes the geometry before scaling; it is not just a unit conversion.

### Follow three actual profiles

The case table below picks a customer near the cohort's feature medians, one with the largest observed positive value, and one with the most observed invoices. These are deterministic examples from this extract, not three known customer types. The same ID could fill two roles. Keep the case table beside your explanation as you inspect neighbors and later groups.

Choose **one** route. Both use log-standardized RFM as the fixed baseline and keep five neighbors.

- **Route A — compare observed purchase value:** ask whether emphasizing monetary value produces neighbors closer in pounds. Change `route` to `'value'`. Inspect monetary differences alongside the sacrifices in recency and frequency. Total positive value does not measure profit.
- **Route B — compare purchase rhythm:** ask whether emphasizing recency and frequency produces neighbors closer on those two features. Change `route` to `'rhythm'`. Inspect original days and invoice counts, and what happens to the monetary differences.

Start with the typical case, then try another case using `query_case`. Predict one result before running. Keep the query and dataset fixed while comparing rulers; changing the query is a second experiment. There is no requirement for every query to change neighbors.

In [ ]:
X_raw = rfm.to_numpy(dtype=float)
X_standard = StandardScaler().fit_transform(X_raw)
X_log = StandardScaler().fit_transform(np.log1p(X_raw))
spaces = {'raw': X_raw, 'standardized': X_standard, 'log_standardized': X_log}
median_position = np.median(X_standard, axis=0)
typical_position = np.argmin(np.linalg.norm(X_standard - median_position, axis=1))
case_ids = {'typical': rfm.index[typical_position],
            'largest_value': rfm.monetary.idxmax(),
            'most_invoices': rfm.frequency.idxmax()}
case_profiles = pd.DataFrame([rfm.loc[identifier].to_dict() for identifier in case_ids.values()],
                             index=pd.Index(case_ids, name='case'))
case_profiles.insert(0, 'CustomerID', list(case_ids.values()))
display(case_profiles.round(2))

# TODO 4: Select 'typical', 'largest_value', or 'most_invoices', then rerun this cell.
query_case = 'typical'
query_id = case_ids[query_case]
query_position = rfm.index.get_loc(query_id)
print('Query case:', query_case, '| ID:', query_id)
display(rfm.loc[[query_id]])

def nearest_profiles(X, position, count=5):
    distance = np.linalg.norm(X - X[position], axis=1)
    distance[position] = np.inf
    positions = np.argsort(distance, kind='stable')[:count]
    return rfm.iloc[positions].assign(distance=distance[positions])

neighbor_sets = {}
for name, X in spaces.items():
    neighbors = nearest_profiles(X, query_position)
    neighbor_sets[name] = set(neighbors.index)
    print(name)
    display(neighbors.round(2))

In [ ]:
# TODO 5: Choose 'value' or 'rhythm'; keep query_case fixed for the comparison.
route = 'value'
route_weights = {'value': np.array([1., 1., 2.]),
                 'rhythm': np.array([2., 2., 1.])}
weights = route_weights[route]
X_weighted = X_log * weights
weighted_table = nearest_profiles(X_weighted, query_position)
weighted_neighbors = set(weighted_table.index)
overlap = len(weighted_neighbors & neighbor_sets['log_standardized']) / 5
print('Route:', route, '| Fraction of baseline neighbors retained:', overlap)
display(weighted_table.round(2))

# Read the consequences in actual days, invoices, and pounds.
baseline_table = nearest_profiles(X_log, query_position)
comparison = pd.DataFrame({
    'baseline_mean_absolute_gap': baseline_table[rfm.columns].sub(rfm.loc[query_id]).abs().mean(),
    'route_mean_absolute_gap': weighted_table[rfm.columns].sub(rfm.loc[query_id]).abs().mean()
})
display(comparison.round(2))
# Weight 2 multiplies the squared-distance contribution by 4.
# Explore a different weight only after recording this baseline comparison.

### Write an explanation someone else could reproduce

Write 100–150 words: identify your query and route, state the baseline and changed weights, describe one neighbor in original units, and explain whether the result fits your question. Use the mean absolute gaps as a descriptive check, not a score to optimize across every query. A smaller average gap can hide a poor match for one individual neighbor; inspect the rows too.

If the five IDs are unchanged, that is a valid finding. Their order or distances can still change. Try the second case and report both outcomes rather than selecting only a dramatic result. Distances calculated under different rulers do not share a common interpretation; compare profiles and membership instead of claiming that one ruler's smaller distance is intrinsically better.

For example, an explanation can say: “I emphasized recency and frequency because the question concerns observed purchase rhythm. The new list included a customer closer in invoice count but farther in pounds. That tradeoff fits this descriptive question, although it does not establish common product interests.” Supply your own actual observations.

**Pause:** what information would you add if “similar” meant buying the same kinds of products? RFM does not contain that information. That question will connect naturally to transaction and text representations later.

## Reading B — Does clustering discover groups or impose them?

**Read and reflect independently for 10–15 minutes.** Keep your nearest-neighbor investigation in mind. Clustering extends the question from one customer's neighbors to a proposed description of the whole cohort.

### From a local comparison to a partition

A neighbor list describes who is close to a chosen query under one ruler. A partition assigns every customer to one group. These are related but different tasks. Two customers can both be fairly near the query while appearing on opposite sides of a group boundary. A group can contain a wide range of behavior even when its median profile sounds distinctive.

Imagine arranging customers along a smooth gradient from small occasional purchasers to frequent large purchasers. There may be no obvious gaps. If we ask k-means for three groups, it will still return three groups. Those groups might offer a convenient summary, but the algorithm has not demonstrated that the retailer has three natural types of customer. The distinction matters when someone starts treating a group name as a fact about each member.

### What k-means is trying to achieve

K-means uses a chosen number of groups, $k$, and minimizes the sum of squared distances to group centers:

$$\sum_{i=1}^{n}\|x_i-\mu_{c_i}\|^2.$$

Here $x_i$ is customer $i$ in our selected feature space, $c_i$ identifies its assigned group, and $\mu_{c_i}$ is that group's centroid, the coordinate-wise mean. The algorithm alternates assigning observations to nearby centers and updating those centers. Initial centers can matter, which is why we use several restarts and later inspect sensitivity to seeds.

The objective favors groups whose observations lie close to their center in squared Euclidean distance. Large deviations have substantial influence. Changing scaling or applying a logarithm changes the objective's meaning even when the customer records remain fixed. Increasing $k$ can improve this objective simply because more centers are available; it does not by itself establish a better interpretation.

The moons example below deliberately contains two curved synthetic groups. A center-based partition tends to cut across those curves. We know the generating labels in this controlled example, so we can discuss whether a method recovers them. We do not have corresponding ground-truth customer segments in the retail extract.

### A different assumption: density connectivity

DBSCAN connects observations through dense local neighborhoods. Its radius, `eps`, specifies how close points must be to count as neighbors under the chosen distance. `min_samples` specifies the neighborhood count for a core point, including the point itself in scikit-learn. Connected core neighborhoods can follow a curve. Nearby border points can join a group; points that cannot join are labeled noise.

This gives us a different notion of a group, with its own limitations. A radius that suits one density may split a sparser group or merge nearby denser groups. Feature scales affect what the radius means. Noise means sparse under this representation and these settings; it does not tell us that a transaction is wrong or a customer is fraudulent. Today the moons are a short comparison of assumptions. Tuning DBSCAN on retail is optional.

### Read group profiles before naming them

For the retail baseline, we will use three k-means groups in log-standardized RFM. The model sees those transformed coordinates. To explain its output, we return to days, invoice counts, and positive recorded pounds, and inspect group sizes and medians. A group with more invoices may reasonably be described as having higher observed purchase frequency. Calling it “loyal” introduces a claim about commitment that our measurements do not establish.

Similarly, a median is a summary, not a description of every member. Follow the case customers from the first investigation and check whether the group's profile helps explain their assignments. Inspect an individual who fits poorly before giving a confident name. Terms such as “high observed positive value” keep the description tied to what was measured.

### A projection is another representation choice

PCA finds linear directions with large variance and lets us display the three transformed features in two coordinates. We fit the clustering in all three RFM dimensions and use PCA afterward for the plot. A displayed separation can help us ask questions, while overlap or distortion can hide relationships from the third dimension. Explained variance tells us how much transformed variance the display retains; it does not validate the groups.

The two coordinates are mixtures of features, not new measurements in pounds or days. Group numbers and axis signs are arbitrary. When comparing plots, look for relationships and refer back to the profile table rather than reading a high horizontal position as automatically meaning a better customer.

### Written checkpoint

Explain why k-means can produce groups on a smooth cloud, and why the moons do not demonstrate that DBSCAN is always the better method.

<details><summary>Self-check — open after writing</summary>

K-means is asked to partition the observations into $k$ groups and optimizes distance to centers even when there are no clear gaps. The moons have a deliberately curved geometry that suits density connectivity at selected settings. Another representation, density pattern, or parameter choice can change that comparison. Useful retail descriptions need inspected profiles and further evidence.

</details>

In [ ]:
moons, true_moon = make_moons(n_samples=300, noise=.06, random_state=SEED)
moon_kmeans = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(moons)
moon_dbscan = DBSCAN(eps=.2, min_samples=5).fit_predict(moons)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, labels, title in zip(axes, [true_moon, moon_kmeans, moon_dbscan],
                           ['Known synthetic groups', 'K-means: compact groups', 'DBSCAN: density connectivity']):
    ax.scatter(moons[:, 0], moons[:, 1], c=labels, cmap='tab10', s=18)
    ax.set_title(title); ax.set_xlabel('x1'); ax.set_ylabel('x2')
plt.tight_layout(); plt.show()
# TODO 6: Change eps to 0.1 and 0.4. Count clusters and noise after each run.
print('DBSCAN clusters:', len(set(moon_dbscan) - {-1}), 'noise points:', (moon_dbscan == -1).sum())

### A convincing partition without planted groups

The next data are synthetic draws from one continuous normal cloud. No group labels or separate components were planted in its generation. Predict whether asking k-means for three groups will still produce three colored regions. Compare the uncolored and colored views using identical axes. An algorithmic boundary can be visually clear without corresponding to a known categorical boundary.

In [ ]:
cloud = np.random.default_rng(SEED).normal(size=(300, 2))
cloud_labels = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(cloud)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].scatter(cloud[:, 0], cloud[:, 1], s=18, color='#777777', alpha=.7)
axes[1].scatter(cloud[:, 0], cloud[:, 1], s=18, c=cloud_labels, cmap='tab10', alpha=.7)
for ax, title in zip(axes, ['One continuous synthetic cloud', 'The same cloud partitioned into three groups']):
    ax.set(title=title, xlabel='Synthetic feature 1', ylabel='Synthetic feature 2')
    ax.set_aspect('equal', adjustable='box')
plt.tight_layout(); plt.show()
print('Assigned group sizes:', np.bincount(cloud_labels))

## Real customer groups — name a profile, then challenge the name

We start with three groups in log-standardized RFM. Inspect their sizes and medians in original units. Write a tentative descriptive name for two groups, then underline any word that goes beyond those measurements. What evidence would “loyal,” “profitable,” or “likely to respond” require?

The case table shows the same customers used in Investigation 1. Explain one assignment using its original profile. The PCA plot displays these customers with larger outlined markers and annotations. Clustering used all three transformed features; the display retains two directions of variance. A case marker helps locate an observation, but its displayed position does not replace its actual profile.

In [ ]:
baseline = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X_log)
labels = baseline.labels_
profile = rfm.assign(cluster=labels).groupby('cluster').median()
profile['customers'] = pd.Series(labels).value_counts().sort_index()
display(profile.round(2))
case_groups = case_profiles.copy()
case_groups['group'] = [labels[rfm.index.get_loc(identifier)] for identifier in case_ids.values()]
display(case_groups.round(2))
pca = PCA(n_components=2)
projection = pca.fit_transform(X_log)
fig, ax = plt.subplots(figsize=(9, 5))
for cluster in sorted(set(labels)):
    mask = labels == cluster
    ax.scatter(projection[mask, 0], projection[mask, 1], s=22, alpha=.6, label=f'Group {cluster}')
for number, identifier in enumerate(dict.fromkeys(case_ids.values())):
    position = rfm.index.get_loc(identifier)
    role = '/'.join(name for name, value in case_ids.items() if value == identifier)
    ax.scatter(*projection[position], s=110, facecolors='none', edgecolors='black', linewidths=1.5)
    ax.annotate(role, projection[position], xytext=(6, 8 + 12 * number), textcoords='offset points', fontsize=9)
ax.set(xlabel='Principal component 1', ylabel='Principal component 2',
       title=f'Projection retains {pca.explained_variance_ratio_.sum():.1%} of transformed variance')
ax.legend(); plt.tight_layout(); plt.show()

## Reading C — What would make you trust a grouping?

**Read and reflect independently for 10–15 minutes.** You have a partition and some possible names. Before defending them, separate three questions: how well groups are separated, how much they change under a relevant perturbation, and whether their descriptions help answer the original question.

### Separation: how does a point fit its group?

A point can be close to others in its own group, close to another group, or isolated from everyone. Silhouette summarizes the first two relationships. For an observation, let $a$ be its average distance to the other members of its own group. Let $b$ be the smallest average distance to any other group. Then

$$s=(b-a)/\max(a,b).$$

If $a=2$ and $b=6$, its silhouette is $4/6$, about 0.67. Its own group is closer on average. If both are 4, the value is zero: the two average distances tie. If $a=6$ and $b=2$, it is about −0.67, suggesting the assigned group is a poor fit under this distance. Values lie between −1 and 1. Scikit-learn assigns zero to a singleton group; when both distances are zero, its convention also avoids an undefined division.

The mean silhouette summarizes the cohort, but can hide problematic individuals. It requires at least two groups and fewer groups than observations. In our small cohort we can calculate it directly; computing many pairwise distances becomes expensive for larger datasets. Most importantly, it uses the ruler we supplied. Comparing silhouettes across different representations changes the measurement itself. A higher score is evidence about internal separation, not proof of a superior business description.

### Agreement: compare relationships, not group numbers

Imagine four customers A, B, C, D. One run labels them $[0,0,1,1]$ and another labels them $[1,1,0,0]$. Every numeric label changed, but the partition is identical: A and B remain together, as do C and D. Counting equal labels would give the wrong answer.

The adjusted Rand index, ARI, compares whether pairs of observations are together or apart in two partitions and adjusts for chance agreement given the group sizes. Identical partitions have ARI 1, values near zero indicate chance-level agreement, and negative values are possible. Both partitions must refer to the same observations in the same order. We keep customer IDs aligned throughout the notebook. ARI establishes agreement between two outputs, not agreement with real customer types.

A whole-cohort score still leaves a practical question: whose relationships changed? For one customer, collect its peers in each partition: all other customers assigned to its group. Peer Jaccard overlap is the size of the intersection divided by the size of the union. If its peers change from $\{B,C\}$ to $\{B,D\}$, the overlap is $1/3$. If both peer sets are empty, we define the overlap as one because they agree; if only one is empty, it is zero. This local description also ignores arbitrary group numbers. It is an aid for inspecting changed group relationships, not a confidence score.

### Stability: what did we perturb?

Changing only the random seed probes sensitivity to initialization. Several similar results mean the optimizer found similar solutions on these fixed features. It does not test the effect of a different sample or different representation. Changing standardization to log-standardization keeps the customers fixed but changes their geometry. That experiment tests whether our description depends on which differences we emphasize.

Sampling sensitivity asks another question. We fit the scaler and k-means again using a subset of customers, then assign the full cohort with that fitted pipeline. This gives a common ID set for comparison with the baseline. The full cohort includes customers used in fitting, so the comparison is descriptive perturbation analysis. It does not estimate future-customer performance or a confidence interval. Refitting the scaler matters because the sample affects the ruler as well as the centers.

Keep the number of groups, seed, and purchase policy fixed when testing one sample fraction. Repeat the experiment several times and describe the variation. A smaller sample can sometimes agree more closely on one run by chance; do not force a monotonic story. A stable partition can also be a stable way of cutting a continuous gradient, as the cloud example illustrated.

### Usefulness: return to the question and records

Suppose one group has more invoices and higher observed positive value. This may help summarize the extract. It does not show the group earns greater profit, will respond to a campaign, or represents the retailer's current customers. Those claims require different measurements, representative data, and possibly an intervention study. Our data are historical and selectively sampled, and our monetary feature does not subtract returns.

A useful conclusion states which description survives the comparison and which uncertainty matters. If the case customer's peers change substantially, examine the new peers in days, invoices, and pounds. Perhaps the alternative emphasizes purchase rhythm over volume. If little changes, describe that specific stability result and its scope. Either result can guide the next investigation.

### Written checkpoint

Explain what each test would establish: a higher silhouette, ARI 1 after label swapping, and similar partitions across five seeds. Then name one claim none of these establishes.

<details><summary>Self-check — open after writing</summary>

Silhouette describes separation under the selected ruler. ARI 1 after a label swap recognizes identical grouping relationships. Seed agreement describes initialization stability for fixed data and representation. None establishes profitable customers, natural categories, future performance, or an effective campaign. Connect any recommendation to the additional evidence it needs.

</details>

In [ ]:
# A worked comparison: labels can change while the partition stays identical.
first = np.array([0, 0, 1, 1])
renamed = np.array([1, 1, 0, 0])
different = np.array([0, 1, 0, 1])
print('Same numeric labels after renaming:', np.mean(first == renamed))
print('ARI after renaming:', adjusted_rand_score(first, renamed))
print('ARI for a different partition:', adjusted_rand_score(first, different))

def peer_comparison(reference, alternative):
    """Compare peer sets on the same ordered rfm cohort; group numbers may differ."""
    reference = np.asarray(reference)
    alternative = np.asarray(alternative)
    assert reference.shape == alternative.shape == (len(rfm),)
    rows = []
    for position, identifier in enumerate(rfm.index):
        before = reference == reference[position]
        after = alternative == alternative[position]
        before[position] = after[position] = False
        intersection = np.count_nonzero(before & after)
        union = np.count_nonzero(before | after)
        rows.append({'CustomerID': identifier, 'baseline_peers': int(before.sum()),
                     'alternative_peers': int(after.sum()),
                     'peer_jaccard': intersection / union if union else 1.})
    return pd.DataFrame(rows).set_index('CustomerID')

def inspect_changed_peers(reference, alternative):
    comparison = peer_comparison(reference, alternative)
    display(comparison.loc[list(dict.fromkeys(case_ids.values()))].round(3))
    changed_id = comparison.peer_jaccard.idxmin()
    position = rfm.index.get_loc(changed_id)
    print('Lowest peer overlap in this comparison:', changed_id)
    display(rfm.loc[[changed_id]].join(comparison).round(3))
    before = reference == reference[position]
    after = alternative == alternative[position]
    status = np.full(len(rfm), 'outside both groups', dtype=object)
    status[before & after] = 'retained peer'
    status[before & ~after] = 'lost peer'
    status[~before & after] = 'new peer'
    status[position] = 'query'
    member_profiles = rfm.assign(relationship=status)
    print('Original-unit medians for changed and retained peer sets:')
    display(member_profiles[member_profiles.relationship != 'outside both groups']
            .groupby('relationship').agg(recency=('recency', 'median'),
                                        frequency=('frequency', 'median'),
                                        monetary=('monetary', 'median'),
                                        customers=('monetary', 'size')).round(2))
    print('Inspect up to three actual records from each relationship:')
    display(member_profiles[member_profiles.relationship != 'outside both groups']
            .groupby('relationship', sort=False).head(3).round(2))
    return comparison

## Independent investigation 2 — Try to break your segmentation (40 minutes)

Spend about 10 minutes inspecting two candidate values of `k`, 20 minutes on **one** challenge route, and 10 minutes preparing evidence for the final memo. Keep the supplied three-group baseline available even if your candidate description uses another `k`; the sensitivity cells compare three groups so that representation or sampling is the controlled change.

**First, name it and challenge it.** Compare the baseline profile with one other candidate. Write a descriptive name for two groups using days, invoices, and pounds. Circle an unsupported interpretation and state what additional evidence it would need. Inspect the case customers beside the medians.

- **Route A — change the representation:** keep the cohort, `k=3`, and seed fixed, then compare log-standardized with standardized RFM. Use ARI and the peer inspection table to identify a concrete consequence. Did the customer's new peers become closer in volume, rhythm, or neither? Inspect actual records before explaining the cause.
- **Route B — change the fitted cohort:** keep `k=3`, seed, and feature recipe fixed. Compare fits on 80% and 60% subsets. Refit preprocessing in each run, inspect the distribution of ARI, and follow changed peers for the first subset. That subset is a reproducible illustration; avoid choosing only the most dramatic run.

Both route cells run as supplied. Focus your written interpretation on your chosen route. The goal is an explicit comparison with a limitation, and an unchanged result can be informative.

In [ ]:
# TODO 7: Inspect k=2..6. Choose two candidates using scores AND original-unit profiles.
scores = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_log)
    scores.append({'k': k, 'silhouette': silhouette_score(X_log, model.labels_),
                   'smallest_group': np.bincount(model.labels_).min()})
display(pd.DataFrame(scores).round(3))
candidate_k = 4
candidate = KMeans(n_clusters=candidate_k, n_init=10, random_state=SEED).fit(X_log)
display(rfm.assign(cluster=candidate.labels_).groupby('cluster').agg(['median', 'size']).round(2))

In [ ]:
# Route A: same ordered customers and k, different representation.
challenge_space = 'standardized'  # Try 'raw' only after recording the main comparison.
representation_labels = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(spaces[challenge_space])
print('Representation:', challenge_space, '| ARI vs log baseline:',
      round(adjusted_rand_score(labels, representation_labels), 3))
representation_peers = inspect_changed_peers(labels, representation_labels)

# Initialization alone: fixed dataset and representation.
seed_agreement = []
for seed in range(5):
    other = KMeans(n_clusters=3, n_init=10, random_state=seed).fit_predict(X_log)
    seed_agreement.append(adjusted_rand_score(labels, other))
print('ARI across seeds:', np.round(seed_agreement, 3))

In [ ]:
# Route B: refit BOTH scaler and clustering on subsets, then predict on the full cohort.
# TODO 8: Change sample_fraction from .8 to .6; the labels update automatically.
sample_fraction = .8
assert 0 < sample_fraction <= 1
rng = np.random.default_rng(SEED)
resample_agreement = []
first_subset_labels = None
for repeat in range(10):
    subset = rng.choice(len(rfm), size=int(sample_fraction * len(rfm)), replace=False)
    scaler = StandardScaler().fit(np.log1p(X_raw[subset]))
    model = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(
        scaler.transform(np.log1p(X_raw[subset])))
    predicted = model.predict(scaler.transform(np.log1p(X_raw)))
    if repeat == 0:
        first_subset_labels = predicted.copy()
    resample_agreement.append(adjusted_rand_score(labels, predicted))
print(f'{sample_fraction:.0%} sample: ARI min / median / max:', np.round([
    min(resample_agreement), np.median(resample_agreement), max(resample_agreement)], 3))
plt.plot(range(1, 11), resample_agreement, 'o-', color='#277da8')
plt.ylim(-.05, 1.05)
plt.xlabel(f'{sample_fraction:.0%} subsample'); plt.ylabel('ARI against full-cohort partition')
plt.title('Descriptive sensitivity, not out-of-sample validation')
plt.tight_layout(); plt.show()
print('Peer inspection for the first subset, rather than a selected extreme run:')
sampling_peers = inspect_changed_peers(labels, first_subset_labels)

### Decision memo and wrap-up

Prepare a short evidence table before writing 150–200 words:

| Claim | Baseline evidence | Your challenge | What changed? | Revised claim or limitation |
|---|---|---|---|---|
| A group description or one customer's relationships | Original-unit profile, size, and inspected example | Named representation or sample change | Metric plus a concrete profile or peer change | What you can still explain and what needs more evidence |

Give your representation and proposed number of groups; characterize two groups in original units; report separation and your chosen stability comparison; and describe one inspected customer's relationships. End with a decision the evidence does not justify and the evidence you would collect next. “No actionable segmentation yet” is acceptable when supported.

If time is limited, prioritize one controlled challenge with an inspected case over more parameter values. Compare explanations with another reader: can they identify your baseline, changed setting, observation unit, and limitation?

**Carry forward:** write one sentence linking Week 1's purchase policy to today's features, and another linking today's representation to an unusual or poorly matched customer. The same questions about units and evidence will matter when we study anomalies and baskets.

The resampling exercise measures sensitivity relative to a full-cohort fit. Its evaluation includes fitted observations, so it is not a future-customer generalization test or a confidence interval. Strong stability also does not prove real natural categories: an algorithm can consistently partition a continuous cloud.

Discuss: could a different distance be more appropriate? Does the projection conceal overlap? Which business evidence would you collect before acting on a segment? Next week we switch from geometric similarity to items that occur together in transactions.

## Reading path

- **Start here · 20–25 min:** [Introduction to Statistical Learning with Python](https://www.statlearning.com/), Chapter 12, introductions to PCA and k-means clustering. Focus on the geometric explanations. Question: why do the units of a feature matter?
- **Go deeper · 20 min:** [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html), k-means, DBSCAN, and silhouette sections. Question: which shapes or density differences would break each method's assumptions?
- **Visual extension · 15–20 min:** [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/), examples on cluster size, distance, and parameter changes. We used PCA today; this prepares you to question nonlinear projection plots later.
- **Try it · 15 min:** repeat our moons experiment at two noise levels, recording DBSCAN's noise count. Explain which change is in the data and which is in the model.

## Homework / Post-class Extensions — optional

1. Compare k-means with DBSCAN on retail features. Report noise separately; calculate silhouette only when at least two non-noise groups and enough observations remain.
2. Inspect PCA loadings. Which transformed RFM features contribute most to each axis? Do not interpret the sign of a PCA axis as an absolute meaning.
3. Add a return-related feature using the retained raw records. Explain how it changes the observation policy and nearest neighbors.
4. Try a user–genre profile from [MovieLens Small](https://grouplens.org/datasets/movielens/). Compare cosine and Euclidean similarity. This optional alternative is not bundled; use the provider documentation and terms if you investigate it separately. The core notebook prepares all required data automatically.